# Week 07 — Python Solution Lab
## Momentum and Collisions

**Companion to `notebooks/Week_07.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_07.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P4` | Perfectly Inelastic Collision | energy-loss audit, reduced mass |
| **L2 · Intermediate** | `P5` | Elastic Collision | mass-ratio sweep, three classic limits |
| **L3 · Challenge** | `P9` | Multi-Body Collision Chain | sequential-collision resolver + termination proof |

---

## L1 · Basic — P4: Perfectly Inelastic Collision

> **Problem (Week_07.ipynb, L1 — P4).** A $1200$ kg car travelling east at $20.0$ m/s collides
> with a $2000$ kg truck travelling east at $8.0$ m/s. They lock together. Find their common
> velocity afterwards.

**Diagram → Principle.** Momentum is conserved (no external horizontal force during the brief
impact). Kinetic energy is **not** — that is what "perfectly inelastic" means.

**Equation.** $v_f = \dfrac{m_1v_1 + m_2v_2}{m_1 + m_2}$.

**Hand prediction.** $(24000 + 16000)/3200 = 12.5$ m/s east.

**What Python adds.** We compute the energy *lost*, which the algebra above quietly discards. That
energy goes mainly into permanent deformation of the vehicles, heat in the deformed metal, sound
and vibration. How it is absorbed matters: a crumple zone that spreads the same energy over a
longer stopping distance *reduces* the forces on the occupants, which is why deformation is
designed in rather than engineered out. We also show the result equals the centre-of-mass
velocity, which is the deeper reason the formula works.

In [ ]:
# ═══ W07 · L1 · P4 — Perfectly inelastic collision: what momentum keeps, energy loses ═══
import numpy as np

# --- MODEL --------------------------------------------------------------
m1, v1 = 1200.0, 20.0      # car,   m/s east
m2, v2 = 2000.0,  8.0      # truck, m/s east

# --- PREDICT: momentum conservation -------------------------------------
p_before = m1*v1 + m2*v2
v_f = p_before / (m1 + m2)
print(f"p_before = {m1*v1:,.0f} + {m2*v2:,.0f} = {p_before:,.0f} kg*m/s")
print(f"v_f = p / (m1+m2) = {v_f:.2f} m/s east")

# --- VERIFY 1: momentum after must equal momentum before ----------------
p_after = (m1 + m2) * v_f
print(f"p_after  = {p_after:,.0f} kg*m/s   -> conserved exactly")
assert np.isclose(p_before, p_after)

# --- VERIFY 2: the centre of mass never noticed the collision -----------
v_cm = (m1*v1 + m2*v2) / (m1 + m2)
print(f"\nv_cm = {v_cm:.2f} m/s -- identical to v_f.")
print("  In a perfectly inelastic collision the objects simply end up moving WITH the")
print("  centre of mass, which never accelerates because the forces are internal.")
assert np.isclose(v_cm, v_f)

# --- What momentum conservation does NOT tell you: the energy -----------
KE_before = 0.5*m1*v1**2 + 0.5*m2*v2**2
KE_after  = 0.5*(m1 + m2)*v_f**2
lost      = KE_before - KE_after
print(f"\nKE before = {KE_before:11,.0f} J")
print(f"KE after  = {KE_after:11,.0f} J")
print(f"KE LOST   = {lost:11,.0f} J   ({100*lost/KE_before:.1f}% of the initial energy)")
print(f"  For scale, that is {lost/4.184e6:.3f} kg of TNT equivalent. Momentum conservation")
print("  says nothing about where it goes: permanent deformation of the structures, heat")
print("  in the deformed metal, sound, and residual vibration all take a share.")

# --- Cross-check the loss against the reduced-mass formula --------------
mu = m1*m2 / (m1 + m2)
lost_formula = 0.5 * mu * (v1 - v2)**2
print(f"\n  reduced-mass check: 1/2 * mu * (v1-v2)^2 = {lost_formula:,.0f} J -> agrees")
assert np.isclose(lost, lost_formula)

# --- CHECK --------------------------------------------------------------
assert abs(v_f - 12.5) < 1e-9
print(f"\n[OK] Matches textbook answer: v_f = {v_f:.1f} m/s east")

## L2 · Intermediate — P5: Elastic Collision

> **Problem (Week_07.ipynb, L2 — P5).** A $3.0$ kg ball moving at $5.0$ m/s has a head-on
> **elastic** collision with a $1.0$ kg ball at rest. Find (a) the final velocities, (b) the KE
> of each before and after, (c) verify momentum and KE are both conserved.

**Diagram → Principle.** Elastic means **both** momentum and kinetic energy are conserved — two
equations, two unknowns.

**Equation.** $v_{1f} = \dfrac{m_1-m_2}{m_1+m_2}v_{1i}$, $v_{2f} = \dfrac{2m_1}{m_1+m_2}v_{1i}$.

**Hand prediction.** $v_{1f} = 2.50$ m/s, $v_{2f} = 7.50$ m/s.

**What Python adds.** Part (c) says "verify" — so we actually do, to machine precision, rather
than nodding at it. Then we sweep the mass ratio and recover all three classic limits in one
plot: equal masses swap velocities, a heavy projectile barely slows, and a light one bounces
straight back. Those limits are worth more than the single numeric answer.

In [ ]:
# ═══ W07 · L2 · P5 — Elastic collision, verified, then generalised over mass ratio ═══
import numpy as np
import matplotlib.pyplot as plt

def elastic_1d(m1, m2, u1, u2=0.0):
    """Closed-form 1D elastic collision."""
    v1 = ((m1 - m2)*u1 + 2*m2*u2) / (m1 + m2)
    v2 = ((m2 - m1)*u2 + 2*m1*u1) / (m1 + m2)
    return v1, v2

# --- MODEL --------------------------------------------------------------
m1, m2, u1 = 3.0, 1.0, 5.0

# --- (a) PREDICT --------------------------------------------------------
v1, v2 = elastic_1d(m1, m2, u1)
print(f"(a) v1f = {v1:.3f} m/s   (still moving forward: m1 > m2)")
print(f"    v2f = {v2:.3f} m/s")

# --- (b) kinetic energies ------------------------------------------------
KE = lambda m, v: 0.5*m*v**2
print(f"\n(b)            before        after")
print(f"    ball 1  {KE(m1,u1):9.3f} J {KE(m1,v1):9.3f} J")
print(f"    ball 2  {KE(m2,0.):9.3f} J {KE(m2,v2):9.3f} J")
print(f"    total   {KE(m1,u1):9.3f} J {KE(m1,v1)+KE(m2,v2):9.3f} J")

# --- (c) VERIFY both conservation laws to machine precision -------------
p_b, p_a = m1*u1, m1*v1 + m2*v2
E_b, E_a = KE(m1,u1), KE(m1,v1) + KE(m2,v2)
print(f"\n(c) momentum: {p_b:.10f} -> {p_a:.10f}   (residual {abs(p_a-p_b):.2e})")
print(f"    energy:   {E_b:.10f} -> {E_a:.10f}   (residual {abs(E_a-E_b):.2e})")
assert abs(p_a - p_b) < 1e-12 and abs(E_a - E_b) < 1e-12
print("    both conserved to machine precision. [verified]")

# --- A third invariant worth knowing: relative velocity reverses --------
print(f"\n    relative velocity before: {u1 - 0:.3f} m/s")
print(f"    relative velocity after:  {v2 - v1:.3f} m/s  -> same magnitude, reversed")
assert np.isclose(v2 - v1, u1)

# --- GENERALISE: sweep the mass ratio -----------------------------------
ratios = np.logspace(-2, 2, 500)                 # m1/m2
V1, V2 = elastic_1d(ratios, 1.0, u1)             # m2 = 1, m1 = ratio
fig, ax = plt.subplots(figsize=(7.4, 4))
ax.semilogx(ratios, V1, color="#1565c0", lw=2, label="$v_{1f}$ (projectile)")
ax.semilogx(ratios, V2, color="#e65100", lw=2, label="$v_{2f}$ (target)")
ax.axhline(0, c="k", lw=.8); ax.axhline(u1, ls=":", c="grey")
ax.axvline(1, ls=":", c="grey")
ax.axvline(m1/m2, ls="--", c="#2e7d32", label=f"this problem, $m_1/m_2$ = {m1/m2:.0f}")
ax.set_xlabel("mass ratio $m_1/m_2$"); ax.set_ylabel("final velocity (m/s)")
ax.set_title("W07 P5 — the three classic elastic limits")
ax.grid(alpha=.3, which="both"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("\n  limits read off the sweep:")
for r, name in ((0.01, "light hits heavy"), (1.0, "equal masses"), (100.0, "heavy hits light")):
    a_, b_ = elastic_1d(r, 1.0, u1)
    print(f"    m1/m2 = {r:6.2f} ({name:17s}): v1f = {a_:+6.2f}, v2f = {b_:+6.2f} m/s")
print("    equal masses swap velocities exactly; a light ball rebounds at nearly -u1;")
print("    a heavy one keeps almost all its speed and kicks the target to nearly 2*u1.")
assert np.allclose(elastic_1d(1.0, 1.0, u1), (0.0, u1))

# --- CHECK --------------------------------------------------------------
assert abs(v1 - 2.5) < 1e-12 and abs(v2 - 7.5) < 1e-12
print(f"\n[OK] Matches textbook answer: v1f = {v1:.2f} m/s, v2f = {v2:.2f} m/s")

## L3 · Challenge — P9: Multi-Body Collision Chain

> **Problem (Week_07.ipynb, L3 — P9).** Three balls on a frictionless line. Ball A ($2.0$ kg)
> moves at $6.0$ m/s toward ball B ($2.0$ kg) at rest, which is near ball C ($4.0$ kg) at rest.
> All collisions are elastic and happen **sequentially**. Find the final velocity of each ball.

**Diagram → Principle.** Apply the elastic pair formula twice, in the right order. Then — and
this is the step most students skip — **check whether any further collision can still occur**.

**Equation.** Apply $v_{1f}, v_{2f}$ pairwise; A→B first, then B→C.

**The obvious answer is incomplete — and this is the point of the problem.** A and B are equal
masses, so they swap: A stops, B moves at $6.0$ m/s. Then B ($2$ kg) hits C ($4$ kg), giving
$v_B = -2.0$ m/s, $v_C = +4.0$ m/s. It is tempting to stop here — the problem statement even says
"(A hits B, then B hits C)". **But B is now travelling backwards at $2.0$ m/s straight into A,
which is sitting at rest right behind it** — so a *third* collision happens, and A and B (equal
masses again) swap once more. (The printed answer key does count this third collision; the
problem statement is what understates it.)

**Correct final state:** $v_A = -2.0$ m/s, $v_B = 0$, $v_C = +4.0$ m/s.

**What Python adds.** A small **sequential-collision resolver for this ordered three-ball
configuration**: repeatedly find an adjacent pair that is still approaching, resolve it, stop when
none are. That loop cannot forget the third collision the way a hand solution does, and its
termination test — velocities non-decreasing from left to right — is a rigorous proof that no
further contact is possible. We audit momentum and energy after *every* event.

> **What this is not.** It tracks velocities only, with no positions or gaps, so it cannot order
> collisions in time. That is fine here because the balls stay in a fixed left-to-right order and
> we always resolve the leftmost pending pair. A genuine event-driven engine — one that handles
> arbitrary spacing, or decides which of two simultaneous pairs collides first — needs positions
> and time-to-contact as well. Do not reuse this loop for that.

In [ ]:
# ═══ W07 · L3 · P9 — Sequential-collision resolver (ordered 3-ball chain) ═══
import numpy as np

def elastic_1d(m1, m2, u1, u2):
    v1 = ((m1 - m2)*u1 + 2*m2*u2) / (m1 + m2)
    v2 = ((m2 - m1)*u2 + 2*m1*u1) / (m1 + m2)
    return v1, v2

# --- MODEL: balls left-to-right, A behind B behind C --------------------
names = ["A", "B", "C"]
m     = np.array([2.0, 2.0, 4.0])
v     = np.array([6.0, 0.0, 0.0])

p0 = np.sum(m*v); E0 = np.sum(0.5*m*v**2)
print(f"initial:  v = {v},  p = {p0:.3f} kg*m/s,  E = {E0:.3f} J\n")

# --- Resolve collisions until no adjacent pair is approaching -----------
event = 0
while True:
    # a pair (i, i+1) collides if the left ball is catching the right one
    approaching = [i for i in range(len(v)-1) if v[i] > v[i+1] + 1e-12]
    if not approaching:
        break
    i = approaching[0]                       # leftmost pending collision
    event += 1
    before = v.copy()
    v[i], v[i+1] = elastic_1d(m[i], m[i+1], v[i], v[i+1])
    print(f"event {event}: {names[i]} hits {names[i+1]}   "
          f"({before[i]:+.2f}, {before[i+1]:+.2f}) -> ({v[i]:+.2f}, {v[i+1]:+.2f}) m/s")
    # audit the conservation laws after EVERY event
    p, E = np.sum(m*v), np.sum(0.5*m*v**2)
    print(f"          p = {p:9.4f} (residual {abs(p-p0):.1e})   "
          f"E = {E:8.4f} (residual {abs(E-E0):.1e})")
    assert abs(p - p0) < 1e-10 and abs(E - E0) < 1e-10
    assert event < 50, "runaway collision loop"

print(f"\nno pair is approaching any more -> the chain is complete after {event} events.")
for n, mi, vi in zip(names, m, v):
    print(f"  {n} ({mi:.1f} kg): {vi:+.3f} m/s")

# --- Why it terminates: check the ordering explicitly -------------------
print(f"\ntermination check (velocities must be non-decreasing left to right):")
print(f"  v_A = {v[0]:+.2f}  <=  v_B = {v[1]:+.2f}  <=  v_C = {v[2]:+.2f}   -> "
      f"{'yes, no further contact possible' if np.all(np.diff(v) >= -1e-12) else 'NO'}")
assert np.all(np.diff(v) >= -1e-12)
print("\n  THE TRAP: after event 2, B is recoiling at -2.00 m/s while A sits at rest")
print("  directly behind it (A is to B's LEFT). So B runs back into A -- event 3 is")
print("  mandatory. Stopping after two collisions gets A and B wrong.")
print("  (The printed answer key does include event 3; only the problem statement,")
print("   '(A hits B, then B hits C)', suggests there are just two.)")
assert event == 3, "the A-B rebound must be counted"

# --- Final audit --------------------------------------------------------
print(f"\nfinal momentum {np.sum(m*v):.4f} kg*m/s  (started {p0:.4f})")
print(f"final energy   {np.sum(0.5*m*v**2):.4f} J       (started {E0:.4f})")

# --- CHECK --------------------------------------------------------------
assert np.allclose(v, [-2.0, 0.0, 4.0]), f"got {v}"
print("\n[OK] Final state: A = -2.0 m/s, B = 0 m/s, C = +4.0 m/s")
print("     Momentum and energy conserved exactly at every step.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_07.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
